In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate as si

def chebD(n):
    """
    Computes the (n+1)x(n+1) spectral differentiation matrix\
    using Chebyshev roots according to Ch 6 of Trefethen's\
    "Spectral methods in MATLAB". Returns D and nodes on [-1, 1].
    Note that they are ordered backwards, i.e. 1 to -1!
    """
    if n == 0:
        x = 1; D = 0; w = 0
    else:
        a = np.linspace(0.0, np.pi, n+1)
        x = np.cos(a)
        b = np.ones_like(x)
        b[0] = 2; b[-1] = 2
        d = np.ones_like(b)
        d[1::2] = -1
        c = b*d
        X = np.outer(x, np.ones(n+1))
        dX = X - X.T
        D = np.outer(c, 1/c) / (dX + np.identity(n+1))
        D = D - np.diag(D.sum(axis=1))
    return D, x

# Spectral solver

om = 100 # frequency parameter

def solve(n):
    """
    Encodes the ODE and initial conditions with (n+1)-node collocation, and solves the\
    rectangular system with numpy's least squares method.
    """
    D, x = chebD(n)
    lhs = np.zeros((n+2, n+1))
    lhs[:-1, :] = D @ D + om**2*(np.identity(n+1))
    lhs[-2, :] = 0; lhs[-2, -1] = 1
    lhs[-1, :] = D[-1]
    rhs = np.zeros(n+2)
    rhs[-2] = 1
    rhs[-1] = 0
    u, res, rank, sing = np.linalg.lstsq(lhs, rhs)
    return u

def solve_spectral():
    """
    Repeatedly calls the spectral solver with double the nodes, comparing the answer at the end of\
    the solution interval until the tolerance is reached, or a certain n is exceeded.
    """
    tol = 1e-12
    n_ini = 16
    n = n_ini
    u = solve(n)
    err = 1
    while err > tol and n <= 256: # The second condition here is just to guard against an infinite loop
        u_new = solve(n+1)
        err = np.abs(u[0] - u_new[0])
        u = u_new; n += 1;
        print(n, err)
    return u[0]
    
%time solve_spectral()

# Scipy's default Runge-Kutta solver

def f_ivp(t, u):
    """ The right-hand-side in u' = f(t, u). """
    du = np.zeros(2)
    du[0] = u[1]
    du[1] = - om**2*u[0]
    return du

u0 = np.array([1, 0])
tspan = (-1, 1)

%time solution = si.solve_ivp(f_ivp, tspan, u0, rtol = 1e-12, atol = 1e-12)

solution.t[-1], solution.y[0, -1]

17 1.2013414457152881e-06
18 8.568190271814627e-07
19 5.594128889839489e-07
20 3.1045499830305246e-07
21 1.1876005914068337e-07
22 5.147668141560907e-09
23 1.3176731305413761e-08
24 2.3630805548557457e-07
25 8.967184839093779e-07
26 2.6295832939072088e-06
27 7.961628758456423e-06
28 3.8420312016787985e-05
29 0.0014043219954597717
30 0.0013899045750185083
31 2.1724910755767024e-05
32 4.553260115710882e-06
33 8.215698590054063e-07
34 4.763494733567651e-07
35 4.354970487777178e-06
36 3.658161198021617e-05
37 0.0006934849825284512
38 0.0006791487259777643
39 2.2696054550605814e-05
40 5.4114489446920455e-06
41 1.1289203110752474e-06
42 3.583893666460773e-07
43 3.0503541190613664e-06
44 1.8942039417869863e-05
45 0.000944530061774086
46 0.0009608303428345864
47 3.595135154185093e-05
48 3.4784946429871496e-06
49 2.5404039129250274e-06
50 3.651486843280158e-05
51 0.0005763485764960464
52 0.0005577147709742073
53 1.7431464142825803e-05
54 2.07034485341695e-06
55 3.1837519165524162e-06
56 3.69243

122 0.13685370846916167
123 0.003728055716073264
124 0.016199633204425024
125 0.000348776983252419
126 0.001226159727230114
127 2.6643079910404488e-05
128 7.96758958283017e-05
129 1.704714620021086e-06
130 4.687813898729409e-06
131 1.0978280345907976e-07
132 2.3492590173201933e-07
133 3.3793016718242086e-09
134 1.334074234415894e-08
135 6.268562335876027e-10
136 1.3882806015885762e-10
137 6.732681079313352e-11
138 8.805089990460147e-11
139 1.786482073384832e-11
140 1.1209422279279124e-11
141 3.552769189951732e-12
142 1.8655077482776505e-12
143 3.6992631180510216e-13
CPU times: user 5.46 s, sys: 2.93 ms, total: 5.47 s
Wall time: 346 ms


CPU times: user 2.38 s, sys: 5.32 ms, total: 2.38 s
Wall time: 640 ms


(np.float64(1.0), np.float64(0.4871876749855047))